# 11 · Model B: AddressBind-style dual encoder (GPU)

**Idea (after Amazon, SIGSPATIAL 2025):** learn one vector space shared by address text and map locations. An address should land next to its own location and far from every other.

```
address text -> MiniLM text encoder -------------> vector \
                                                           cosine similarity, contrastive (InfoNCE) loss
(lat, lon)   -> Fourier features -> small MLP ----> vector /
```

**Geocoding** = embed every candidate point in the city once (the centre of each H3 resolution-10 cell that contains a building), embed the address, return the best-scoring point.

**Why it might beat the classifier on cold areas:** the location encoder is a smooth function of position, so it can score places it never saw. How well it does so is the experiment.

**Second use (later, P3):** the same similarity score says how well a given pin matches a given address, which is the wrong-pin detector.

**Runtime:** about 25 minutes on a T4 for 4 epochs. **Needs:** notebooks 01 and 06. Checked on CPU with a tiny random network; not yet trained on a GPU.

In [ ]:
# Setup (the same cell opens every notebook): fetch the project code, install what is missing, find the data folder.
import os, sys, subprocess
REPO = 'SiddharthaRaja/Navigate'
PIP  = ['h3', 'rapidfuzz', 'transformers>=4.44']
on_cloud = os.path.exists('/content') or os.path.exists('/kaggle/working')
if on_cloud:
    CODE = '/kaggle/working/Navigate-code' if os.path.exists('/kaggle/working') else '/content/Navigate'
    if not os.path.exists(f'{CODE}/bench'):
        token = os.environ.get('GH_TOKEN', '')
        try:                                   # only needed while the GitHub repo is private
            from google.colab import userdata; token = token or userdata.get('GH_TOKEN')
        except Exception: pass
        try:
            from kaggle_secrets import UserSecretsClient; token = token or UserSecretsClient().get_secret('GH_TOKEN')
        except Exception: pass
        url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
        r = subprocess.run(['git', 'clone', '--depth', '1', url, CODE], env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
        assert r.returncode == 0, 'Could not fetch the code. If the repo is private, add a GitHub token as the secret GH_TOKEN.'
    else:
        subprocess.run(['git', '-C', CODE, 'pull', '--ff-only'])
    if PIP: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PIP], check=True)
else:
    CODE = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
sys.path.insert(0, CODE)
import numpy as np, pandas as pd
from bench import env
P = env.setup('hyderabad')
print('running on', P['where'], '| data folder:', P['root'])

import torch
assert torch.cuda.is_available() or P['where'] == 'local', 'No GPU: Runtime > Change runtime type > T4 GPU (Colab) or Settings > Accelerator (Kaggle)'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU trial run)')

In [ ]:
from bench import io, metrics as mt
from bench.geo import CITY_CENTRE
from models import common, addressbind as ab
common.set_seed(7)
ENCODER, EPOCHS = 'sentence-transformers/all-MiniLM-L6-v2', 4
TRAIN_ROWS = None
centre = CITY_CENTRE['hyderabad']
train, val, test = common.load_bench(P['bench'], TRAIN_ROWS)
print(f"{len(train):,} training addresses")

## Train
Each step, every address must pick its own location out of the other 255 in the batch plus 512 random points (half of them within about a kilometre: near misses teach fine position). The temperature is learned.

In [ ]:
model, tok = ab.train_model(train, centre, encoder_name=ENCODER, epochs=EPOCHS, batch_size=256, n_extra=512)

## Candidate points: every H3 resolution-10 cell that contains a building
This uses all buildings in the city, including cold areas. That is fair: a deployed system has the map; what it lacks is addresses for the new area.

In [ ]:
b = pd.read_parquet(f"{P['gaz']}/buildings.parquet", columns=['h3_10', 'lat', 'lon'])
cand = b.groupby('h3_10')[['lat', 'lon']].mean().reset_index()
cand_emb = ab.embed_candidates(model, cand.lat, cand.lon, centre)
print(f"{len(cand):,} candidate points embedded")

In [ ]:
pred, topk = ab.predict(model, tok, test, cand_emb, cand.lat.to_numpy(), cand.lon.to_numpy())
io.save_run(P['runs'], 'addressbind', pred, about=f'Dual encoder ({ENCODER} + Fourier-feature location encoder), contrastive training, retrieval over H3-10 cell centres.')
_, res = mt.evaluate(test, pred, 'addressbind', slices=False)
print(mt.leaderboard(res).to_string())
for sp in ['test_warm', 'test_cold']:
    m = (test.split == sp).to_numpy() & ~test.abstain.to_numpy()
    print(sp, 'retrieval:', ab.hit_at_k(test[m], topk[m], cand.lat, cand.lon, radius_m=100, ks=(1, 5)))
print(f"speed: {pred.latency_ms.median():.2f} ms per address (batched)")
torch.save({'state': model.state_dict(), 'encoder': ENCODER, 'centre': centre}, f"{P['runs']}/addressbind/model.pt")

## Share the outputs
Packs only the files this notebook created into one zip in `My Drive > Navigate > data > share`. Download that one file to share or back up the results.

In [ ]:
env.pack_outputs(P, '11' + '_outputs', [f"{P['runs']}/addressbind"])